In [1]:
import os
import statistics
from collections import Counter
from datetime import datetime, timezone
from json import dumps
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import dotenv_values


In [2]:
# 1. Chargement ciblé des variables d'environnement
env_vars = dotenv_values(".env")
env_local_vars = dotenv_values(".env.local")

atlas_uri = env_vars.get("ATLAS_URI")
local_uri = env_local_vars.get("LOCAL_URI")

# 3. Test de la connexion cloud (Atlas)
try:
    if not atlas_uri:
        raise ValueError("ATLAS_URI non trouvé dans .env")
    client = MongoClient(atlas_uri, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    db = client["securite_routiere"]
    print("Connexion cloud établie avec succès")
except Exception as e:
    print(f"Erreur de connexion cloud : {e}")

Connexion cloud établie avec succès


In [ ]:
# Récupération des 5 premiers éléments de la collection "accidents" pour vérification
try:
    accidents_collection = db["accidents"]
    sample_accidents = accidents_collection.find().limit(5)
    print("Exemple d'accidents récupérés :")
    for accident in sample_accidents:
        print(accident)
except Exception as e:
    print(f"Erreur lors de la récupération des accidents : {e}")

## Livrable 4 — Index justifiés et mesurés

Les quatre requêtes sont testées avec la même méthode :

- **avant** : `hint={"$natural": 1}` force un parcours de collection ;
- **après** : `hint` force l'index testé.

Cela permet de comparer les plans sans supprimer les index de la base partagée.


In [ ]:
# Requêtes utilisées pour les quatre tests
queries = {
    "Num_Acc": {
        "filter": {"Num_Acc": 202400034184},
        "index": "idx_num_acc"
    },
    "Département + météo": {
        "filter": {"dep": "75", "atm": 2},
        "index": "idx_dep_atm"
    },
    "Gravité des usagers": {
        "filter": {"vehicules.usagers.grav": 2},
        "index": "idx_usagers_gravite"
    },
    "Localisation": {
        "filter": {
            "localisation": {
                "$geoWithin": {
                    "$centerSphere": [
                        [2.3522, 48.8566],
                        5 / 6378.1
                    ]
                }
            }
        },
        "index": "idx_localisation_2dsphere"
    }
}

def explain_query(filter_query, hint):
    return db.command(
        "explain",
        {
            "find": "accidents",
            "filter": filter_query,
            "hint": hint
        },
        verbosity="executionStats"
    )

def resume_explain(explain):
    stats = explain["executionStats"]
    return {
        "documents_retournes": stats["nReturned"],
        "documents_examines": stats["totalDocsExamined"],
        "cles_examinees": stats["totalKeysExamined"],
        "temps_ms": stats["executionTimeMillis"]
    }

# Mesure avant index : on force volontairement un COLLSCAN.
before_results = {}

for label, test in queries.items():
    explain = explain_query(test["filter"], {"$natural": 1})
    before_results[label] = resume_explain(explain)

pd.DataFrame(before_results).T


In [ ]:
# Création des index utilisés dans le livrable
try:
    # Num_Acc sert à retrouver directement un accident.
    # L'index reste non unique pendant les tests car des doublons ont été ajoutés dans la collection.
    accidents_collection.create_index(
        [("Num_Acc", 1)],
        name="idx_num_acc"
    )

    # On filtre aussi les accidents par département et condition météo.
    # dep est placé en premier pour que le préfixe de l'index reste utilisable seul.
    accidents_collection.create_index(
        [("dep", 1), ("atm", 1)],
        name="idx_dep_atm"
    )

    # Les usagers sont imbriqués dans les véhicules.
    # MongoDB crée donc ici un index multikey.
    accidents_collection.create_index(
        [("vehicules.usagers.grav", 1)],
        name="idx_usagers_gravite"
    )

    # localisation est stocké en GeoJSON.
    # 2dsphere est adapté aux recherches géographiques autour d'un point ou dans une zone.
    accidents_collection.create_index(
        [("localisation", "2dsphere")],
        name="idx_localisation_2dsphere"
    )

    print("Index créés avec succès")

except Exception as e:
    print(f"Erreur lors de la création des index : {e}")

# Vérification des index présents
for index in accidents_collection.list_indexes():
    print(index["name"], index["key"])


In [ ]:
# Mesure après index : on force cette fois l'index correspondant à chaque requête.
after_results = {}

for label, test in queries.items():
    explain = explain_query(test["filter"], test["index"])
    after_results[label] = resume_explain(explain)

pd.DataFrame(after_results).T


In [ ]:
# Comparaison avant / après
comparaison = []

for label in queries:
    avant = before_results[label]
    apres = after_results[label]

    reduction = (
        1 - apres["documents_examines"] / avant["documents_examines"]
    ) * 100

    comparaison.append({
        "test": label,
        "retournes": apres["documents_retournes"],
        "docs_avant": avant["documents_examines"],
        "docs_apres": apres["documents_examines"],
        "cles_avant": avant["cles_examinees"],
        "cles_apres": apres["cles_examinees"],
        "temps_avant_ms": avant["temps_ms"],
        "temps_apres_ms": apres["temps_ms"],
        "reduction_docs_%": round(reduction, 2)
    })

comparaison_df = pd.DataFrame(comparaison)
comparaison_df


### Justification des quatre index

**1. `idx_num_acc` — `{ Num_Acc: 1 }`**  
La requête cherche un accident précis à partir de son numéro. Sans index, MongoDB doit parcourir toute la collection. Avec l'index, il peut atteindre directement le document concerné.

**2. `idx_dep_atm` — `{ dep: 1, atm: 1 }`**  
Cet index sert aux recherches d'accidents dans un département sous une condition atmosphérique donnée. `dep` est placé en premier pour que l'index reste aussi utile pour une recherche par département seul.

**3. `idx_usagers_gravite` — `{ "vehicules.usagers.grav": 1 }`**  
Les usagers sont stockés dans des tableaux imbriqués dans les véhicules. Cet index est donc multikey et permet de filtrer directement les accidents selon la gravité d'au moins un usager.

**4. `idx_localisation_2dsphere` — `{ localisation: "2dsphere" }`**  
Le champ `localisation` contient les coordonnées de l'accident au format GeoJSON. L'index `2dsphere` sert aux recherches géographiques, par exemple pour retrouver les accidents dans un rayon de 5 km autour du centre de Paris.

Dans la requête utilisée ici, les coordonnées sont données dans l'ordre **longitude, latitude** et le rayon de 5 km est converti en radians avec `5 / 6378.1`, comme demandé par `$centerSphere`.

### Point connu pendant les tests

`Num_Acc` devrait identifier un accident, mais des doublons ont été ajoutés pendant les tests de l'équipe. L'index `idx_num_acc` est donc utilisé ici comme index de performance et reste temporairement non unique.
